# GPU pilotu

Çalışma zamanı: **L4 GPU** (tüm eğitimler aynı GPU tipinde yapılacak). Kalıcı her şey Google Drive'da `MyDrive/PraNet_proje/` altında tutulur; oturum kapansa da kaybolmaz.

| Bölüm | Amaç |
|---|---|
| 1 | Drive bağlantısı, kod, paketler, GPU bilgisi |
| 2 | Omurga ağırlıkları + dört modelin yayınlanmış ağırlıkları |
| 3 | Polip veri kümesi ve görüntü sayılarının kontrolü |
| 4 | Ölçüm hattının yayınlanmış ağırlıklarla doğrulanması (makale değerleri + iki çıkarım protokolü) |
| 5 | DSRA testi bu kez **eğitilmiş** ağırlıklarla |
| 6 | Süre ölçümü: cuDNN kapalı/açık, Res2Net ve PVT için birer epoch; ayarın sabitlenmesi |
| 7 | İlk gerçek koşu: PraNet-V2, seed 1 (29 epoch) ve değerlendirmesi |

Bölümler tekrar çalıştırılabilir: tamamlanmış adımlar (indirilmiş dosyalar, biten değerlendirmeler, biten eğitim) atlanır, yarım kalan eğitim kaldığı yerden devam eder.

## 1. Kurulum

Drive'ı bağlar, proje klasörlerini oluşturur, `proje` dalını klonlar (ya da günceller), paketleri yükler ve GPU'yu yazdırır. `run()` yardımcı fonksiyonu betikleri alt süreçte çalıştırır, çıktıyı canlı gösterir ve isteğe bağlı olarak Drive'daki bir log dosyasına da yazar.

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/PraNet_proje'
GITHUB_USER = 'esmapasaoglu'
BRANCH = 'proje'

import os, sys, json, shutil, subprocess, platform, time, glob, zipfile
from google.colab import drive
drive.mount('/content/drive')
for sub in ['data_raw', 'models', 'released', 'runs', 'results', 'logs']:
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)

REPO_DIR = '/content/PraNet-V2'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH,
                    f'https://github.com/{GITHUB_USER}/PraNet-V2.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(f'{REPO_DIR}/binary_seg')
sys.path.insert(0, os.getcwd())
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

In [ ]:
!pip -q install timm thop tabulate gdown
import torch, numpy as np
assert torch.cuda.is_available(), 'GPU yok: Çalışma zamanı türünü L4 GPU yapın'
GPU = torch.cuda.get_device_name(0)
print('Python', platform.python_version(), '| torch', torch.__version__, '| GPU', GPU)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

def run(args, tail=None, log_file=None):
    '''Betiği alt süreçte çalıştırır. tail=None: çıktıyı canlı gösterir; tail=n: yalnızca son n satırı.
    log_file verilirse çıktı oraya da eklenir. Hata olursa son satırları gösterip durur.'''
    cmd = [sys.executable, '-W', 'ignore', '-u'] + args
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    f = open(log_file, 'a') if log_file else None
    for line in p.stdout:
        lines.append(line)
        if f:
            f.write(line); f.flush()
        if tail is None:
            print(line, end='')
    p.wait()
    if f:
        f.close()
    if tail:
        print(''.join(lines[-tail:]), end='')
    if p.returncode != 0:
        if tail is not None:
            print(''.join(lines[-30:]))
        raise RuntimeError('Komut hata verdi (kod %d): %s' % (p.returncode, ' '.join(args)))
    return lines

## 2. Ağırlıklar

Resmî README'deki Google Drive bağlantılarından indirilir ve Drive'daki proje klasöründe saklanır (bir kez indirilir):
- ImageNet omurgaları (Res2Net-50, PVTv2-B2): `binary_seg/models/` klasörüne kopyalanır.
- Dört modelin yazarlar tarafından yayınlanan eğitilmiş ağırlıkları: yalnızca ölçüm hattını doğrulamak için.

In [ ]:
import gdown
FILES = {
    'models/pvt_v2_b2.pth': '1FOv-3T91gJoB153XDUTAg4E0CGPHVQS4',
    'models/res2net50_v1b_26w_4s-3cf99910.pth': '1DzQfXikcSObsL98RmFNHR_7Wuy1vED8z',
    'released/RES-V1.pth': '1VySNrg01d_ydvIKWuAYRPUDMFU1nbnKl',
    'released/RES-V2.pth': '1MFPcbD4Lt3o4d9RAwsT4wfLT32KtLqmF',
    'released/PVT-V1.pth': '1v9c9HGayqsXKpmtIzydMu4h-t9hPwL5y',
    'released/PVT-V2.pth': '1y5i06UYyESxQfawj9swNhyQcyPX6OD1_',
}
for rel, file_id in FILES.items():
    dst = f'{DRIVE_ROOT}/{rel}'
    if not os.path.exists(dst):
        gdown.download(id=file_id, output=dst, quiet=True)
    print(f'{rel:<45} {os.path.getsize(dst) / 1e6:7.1f} MB')
os.makedirs('models', exist_ok=True)
for name in ['pvt_v2_b2.pth', 'res2net50_v1b_26w_4s-3cf99910.pth']:
    shutil.copy(f'{DRIVE_ROOT}/models/{name}', f'models/{name}')
print('Omurgalar binary_seg/models/ klasörüne kopyalandı.')

## 3. Veri kümesi

Resmî veri klasörü bir kez Drive'a indirilir (`data_raw/`). Her oturumda hız için yerel diske (`/content/data`) açılır; eğitim sırasında Drive'dan okumak yavaştır.

**Kontrol:** eğitim kümesi 1450 görüntü olmalı (Kvasir 900 + CVC-ClinicDB 550); test kümeleri Kvasir 100, CVC-ClinicDB 62, CVC-ColonDB 380, ETIS 196, CVC-300 60. Sayılar uymazsa hücre durur.

In [ ]:
DATA_FOLDER_ID = '1Hi-ztnzRtWvdGejYHTyOAn6fNBWcWLik'
RAW = f'{DRIVE_ROOT}/data_raw'
LOCAL = '/content/data'

if not os.listdir(RAW):
    gdown.download_folder(id=DATA_FOLDER_ID, output=RAW, quiet=False)
print('Drive data_raw içeriği:')
for p in sorted(glob.glob(f'{RAW}/**/*', recursive=True))[:30]:
    if os.path.isfile(p):
        print(f'   {os.path.relpath(p, RAW):<50} {os.path.getsize(p) / 1e6:8.1f} MB')

def find_dir(root, name):
    for dirpath, dirnames, _ in os.walk(root):
        if name in dirnames:
            return os.path.join(dirpath, name)
    return None

os.makedirs(LOCAL, exist_ok=True)
if find_dir(LOCAL, 'TrainDataset') is None or find_dir(LOCAL, 'TestDataset') is None:
    for z in sorted(glob.glob(f'{RAW}/**/*.zip', recursive=True)):
        print('Açılıyor:', os.path.basename(z))
        zipfile.ZipFile(z).extractall(LOCAL)
    for name in ['TrainDataset', 'TestDataset']:      # zip yerine klasör geldiyse kopyala
        if find_dir(LOCAL, name) is None and find_dir(RAW, name) is not None:
            shutil.copytree(find_dir(RAW, name), f'{LOCAL}/{name}')
TRAIN = find_dir(LOCAL, 'TrainDataset')
TEST = find_dir(LOCAL, 'TestDataset')
assert TRAIN and TEST, 'TrainDataset/TestDataset bulunamadı; data_raw içeriğine bakın'

def count(d):
    return len([f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.png', '.tif'))])
n_train = count(f'{TRAIN}/images')
print(f'\nTrainDataset: {n_train} görüntü, {count(TRAIN + "/masks")} maske  ({TRAIN})')
EXPECTED = {'Kvasir': 100, 'CVC-ClinicDB': 62, 'CVC-ColonDB': 380, 'ETIS-LaribPolypDB': 196, 'CVC-300': 60}
ok = n_train == 1450
for ds, n in EXPECTED.items():
    d = f'{TEST}/{ds}'
    ni = count(f'{d}/images') if os.path.isdir(d) else 0
    nm = count(f'{d}/masks') if os.path.isdir(d) else 0
    flag = 'ok' if ni == nm == n else 'UYUMSUZ'
    ok = ok and flag == 'ok'
    print(f'Test {ds:<18} {ni:4d} görüntü / {nm:4d} maske (beklenen {n})  {flag}')
assert ok, 'Görüntü sayıları beklenenle uyuşmuyor'
print('\nTAMAM: Veri kümesi beklenen boyutlarda.')

## 4. Ölçüm hattının doğrulanması (yayınlanmış ağırlıklar)

Dört modelin yayınlanmış ağırlıkları `eval_per_image.py` ile beş test kümesinde değerlendirilir. **Resmî protokol** (V1: son harita, V2: dört haritanın toplamı) ile ölçülen mDice'ın makaledeki Kvasir ve ETIS değerlerine yakın çıkması, veri + son işleme + metrik zincirinin doğru kurulduğunu gösterir. Bu kontrol yalnızca ölçüm hattı içindir; kendi eğitimlerimizin sonuçları bu amaçla kullanılmaz.

Aynı ileri geçişte diğer protokol de ölçülür (`last` ve `sum` sütunları). Bu, protokol farkının ek eğitim gerektirmeyen ilk ölçümüdür.

In [ ]:
PAPER = {('PraNet-V1', 'Kvasir'): 89.39, ('PraNet-V1', 'ETIS-LaribPolypDB'): 62.75,
         ('PraNet-V2', 'Kvasir'): 90.70, ('PraNet-V2', 'ETIS-LaribPolypDB'): 64.05,
         ('PVT-PraNet-V1', 'Kvasir'): 87.09, ('PVT-PraNet-V1', 'ETIS-LaribPolypDB'): 68.32,
         ('PVT-PraNet-V2', 'Kvasir'): 91.52, ('PVT-PraNet-V2', 'ETIS-LaribPolypDB'): 76.35}
RELEASED = {'PraNet-V1': 'RES-V1.pth', 'PraNet-V2': 'RES-V2.pth',
            'PVT-PraNet-V1': 'PVT-V1.pth', 'PVT-PraNet-V2': 'PVT-V2.pth'}
summaries = {}
for mt, fname in RELEASED.items():
    out = f'{DRIVE_ROOT}/results/released/{mt}'
    if not os.path.exists(f'{out}/summary.json'):
        print('=' * 12, mt, '=' * 12)
        run(['eval_per_image.py', '--model_type', mt, '--ckpt', f'{DRIVE_ROOT}/released/{fname}',
             '--out_dir', out, '--data_root', TEST], tail=8)
    summaries[mt] = json.load(open(f'{out}/summary.json'))

print(f"\n{'model':<15}{'küme':<19}{'resmî':>8}{'last':>8}{'sum':>8}{'makale':>8}{'fark':>7}")
max_gap = 0
for mt, s in summaries.items():
    off = s['official_protocol']
    for ds, m in s['datasets'].items():
        o, l, su = 100 * m['meanDic_' + off], 100 * m['meanDic_last'], 100 * m['meanDic_sum']
        pap = PAPER.get((mt, ds))
        gap = '' if pap is None else f'{o - pap:+.2f}'
        if pap is not None:
            max_gap = max(max_gap, abs(o - pap))
        print(f"{mt:<15}{ds:<19}{o:8.2f}{l:8.2f}{su:8.2f}{(f'{pap:.2f}' if pap else ''):>8}{gap:>7}")
    if s['load_report']['missing_keys'] or s['load_report']['unexpected_keys']:
        print('   anahtar raporu:', {k: len(v) for k, v in s['load_report'].items()})
print(f'\nMakale ile en büyük fark: {max_gap:.2f} puan')
print('TAMAM: ölçüm hattı makaleyle uyumlu.' if max_gap < 1.0 else
      'DİKKAT: makaleden 1 puandan büyük fark var; bölme/son işleme incelenmeli.')

## 5. DSRA testi, eğitilmiş ağırlıklarla

CPU testinde decoder rastgele başlatılmıştı. Burada aynı test yazarların eğittiği V2 ağırlıklarıyla tekrarlanır. Beklenen yine **CONFIRMED** (resmî ayarda map2–map4 değişimi 0).

In [ ]:
for mt, fname in [('PraNet-V2', 'RES-V2.pth'), ('PVT-PraNet-V2', 'PVT-V2.pth')]:
    print('=' * 12, mt, '(yayınlanmış ağırlıklar)', '=' * 12)
    run(['check_dsra_noop.py', '--model_type', mt, '--ckpt', f'{DRIVE_ROOT}/released/{fname}'], tail=14)
    print()

## 6. Süre ölçümü ve cuDNN kararı

Resmî kod cuDNN'i kapatıyor; bu yalnızca uygulama ayrıntısıdır (tarif değil) ama eğitimi yavaşlatır. PraNet-V2 için birer epoch cuDNN kapalı ve açık ölçülür.

**Önceden yazılmış karar kuralı:** açık ayar en az %20 hızlıysa bütün koşularda `--cudnn 1`, değilse resmî ayar `--cudnn 0` kullanılır. Karar `config.json` dosyasına yazılır ve sonraki tüm koşular bu dosyadan okur; böylece bütün hücreler aynı ayarla eğitilir.

Ardından seçilen ayarla PVT için de bir epoch ölçülür ve 19 koşunun toplam GPU süresi tahmin edilir. Bu ölçüm koşuları geçici klasöre yazılır, analizde kullanılmaz.

In [ ]:
CONFIG = f'{DRIVE_ROOT}/config.json'

def one_epoch(mt, cudnn):
    out = f'/content/timing/{mt}_cudnn{cudnn}'
    shutil.rmtree(out, ignore_errors=True)
    run(['train_seed.py', '--model_type', mt, '--seed', '1', '--out_dir', out, '--train_path', TRAIN,
         '--max_epochs_this_session', '1', '--save_every', '0', '--cudnn', str(cudnn)], tail=2)
    log = json.load(open(f'{out}/train_log.json'))
    return log['epochs'][0]['seconds'], torch.cuda.max_memory_allocated()

if not os.path.exists(CONFIG):
    t0, _ = one_epoch('PraNet-V2', 0)
    t1, _ = one_epoch('PraNet-V2', 1)
    cudnn = 1 if t0 / t1 >= 1.2 else 0
    t_pvt, _ = one_epoch('PVT-PraNet-V2', cudnn)
    t_res = t1 if cudnn else t0
    cfg = {'gpu': GPU, 'cudnn': cudnn, 'epoch_seconds': {'PraNet-V2_cudnn0': t0, 'PraNet-V2_cudnn1': t1,
           f'PVT-PraNet-V2_cudnn{cudnn}': t_pvt}, 'decided': time.strftime('%Y-%m-%d %H:%M')}
    json.dump(cfg, open(CONFIG, 'w'), indent=1)
cfg = json.load(open(CONFIG))
CUDNN = cfg['cudnn']
es = cfg['epoch_seconds']
t_res = es[f'PraNet-V2_cudnn{CUDNN}']
t_pvt = es[f'PVT-PraNet-V2_cudnn{CUDNN}']
print(json.dumps(cfg, indent=1))
print(f"\ncuDNN kapalı/açık hız oranı: {es['PraNet-V2_cudnn0'] / es['PraNet-V2_cudnn1']:.2f}x  ->  karar: --cudnn {CUDNN}")
h_res, h_pvt = 29 * t_res / 3600, 29 * t_pvt / 3600
print(f'Bir koşu (29 epoch): Res2Net ~{h_res:.2f} sa, PVT ~{h_pvt:.2f} sa')
print(f'Toplam 19 koşu (10 Res2Net + 9 PVT): ~{10 * h_res + 9 * h_pvt:.1f} GPU-saat')
if GPU != cfg['gpu']:
    print(f"DİKKAT: bu oturumun GPU'su ({GPU}) kararın verildiği GPU'dan ({cfg['gpu']}) farklı!")

## 7. İlk gerçek koşu: PraNet-V2, seed 1

Bu, ana deneyin ilk koşusudur (pilot ayrıca harcanmaz). 29 epoch, resmî tarif, `config.json`'daki cuDNN ayarı. Ağırlıklar ve log Drive'a yazılır (`runs/PraNet-V2_s1/`). Oturum kesilirse hücreyi yeniden çalıştırmak kaldığı epoch'tan devam ettirir. Eğitim bitince model beş test kümesinde değerlendirilir; bu test kümelerine yapılan **ilk ve tek** bakıştır.

In [ ]:
MT, SEED = 'PraNet-V2', 1
RUN = f'{DRIVE_ROOT}/runs/{MT}_s{SEED}'
if not os.path.exists(f'{RUN}/last.pth'):
    run(['train_seed.py', '--model_type', MT, '--seed', str(SEED), '--out_dir', RUN,
         '--train_path', TRAIN, '--save_every', '0', '--cudnn', str(CUDNN)],
        log_file=f'{DRIVE_ROOT}/logs/{MT}_s{SEED}.txt')
log = json.load(open(f'{RUN}/train_log.json'))
print(f"\nEğitim süresi: {log['total_train_seconds'] / 3600:.2f} sa | GPU: {log['gpu']}")

EVAL = f'{DRIVE_ROOT}/results/runs/{MT}_s{SEED}'
if not os.path.exists(f'{EVAL}/summary.json'):
    run(['eval_per_image.py', '--model_type', MT, '--ckpt', f'{RUN}/last.pth', '--out_dir', EVAL,
         '--data_root', TEST], tail=8)
s = json.load(open(f'{EVAL}/summary.json'))
print(f"\n{'küme':<19}{'mDice (sum, resmî)':>20}{'mDice (last)':>14}")
for ds, m in s['datasets'].items():
    print(f"{ds:<19}{100 * m['meanDic_sum']:20.2f}{100 * m['meanDic_last']:14.2f}")